In [1]:
import os
import warnings
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.model_selection import LeaveOneGroupOut, ParameterGrid
from sklearn.metrics import f1_score, balanced_accuracy_score, recall_score, precision_score, confusion_matrix
from xgboost import XGBClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import catboost as cb
from catboost import CatBoostClassifier
from sklearn.svm import SVR, SVC
from lightgbm import LGBMClassifier
from scikeras.wrappers import KerasClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

In [2]:
def create_sequences_by_participant(df, window_size, feature_cols, target_col):
    X_list, y_list, groups_list = [], [], []
    
    for p_id, group in df.groupby("participant"):
        # Se il paziente ha abbastanza dati per almeno una finestra
        if len(group) >= window_size:
            features = group[feature_cols].values
            target = group[target_col].values
            
            # Crea sequenze solo per questo paziente
            for i in range(len(group) - window_size):
                X_list.append(features[i : i + window_size])
                y_list.append(target[i + window_size])
                groups_list.append(p_id) # Salva il partecipante corretto
                
    return np.array(X_list), np.array(y_list), np.array(groups_list)

In [4]:
def build_lstm_model(n_features, timesteps=1, n_units=64, dropout=0.2, lr=1e-3):

    model = Sequential([
        LSTM(64, input_shape=(timesteps, n_features)),
        Dense(32, activation="relu"),
        Dense(1, activation="sigmoid")
    ])
    
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    #model.summary()
    return model

In [6]:
# Suppress warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
seed = 69
torch.manual_seed(seed)
np.random.seed(seed)

# Define root directory
root = '.'

In [7]:
df = pd.read_csv('./new_dataset/maison-llf-features.CSV', sep=",")

In [9]:
ana = pd.read_csv('./new_dataset/maison-llf-demographics.CSV', sep=",")

In [10]:
ana_col = list(ana.columns)

In [14]:
ana_encoded = ana[["participant", "age", "sex", "education", "work", "fracture-type", "ethnicity"]].copy()

# male=1, female=0
ana_encoded["sex_male"] = ana_encoded["sex"].map({"male": 1, "female": 0})

# Education label encoding with doctorate as highest level
education_order = {
    "secondary education": 0,
    "undergraduate degree": 1,
    "graduate degree": 2,
    "doctorate degree": 3
}
ana_encoded["education_label"] = ana_encoded["education"].map(education_order)

# retired=0, employed part-time=1
ana_encoded["work_part_time"] = ana_encoded["work"].map({"retired": 0, "employed part-time": 1})

fracture_dummies = pd.get_dummies(
    ana_encoded["fracture-type"],
    prefix="fracture",
    dtype=int
)

ethnicity_dummies = pd.get_dummies(
    ana_encoded["ethnicity"],
    prefix="ethnicity",
    dtype=int
)

num_ana = pd.concat(
    [
        ana_encoded[["participant", "age", "sex_male", "education_label", "work_part_time"]],
        fracture_dummies,
        ethnicity_dummies,
    ],
    axis=1,
)

data = df.merge(num_ana, how='right', on='participant')

In [15]:
data.head(5)

,participant,timestamp,clinical-timestamp,sis-01,sis-02,sis-03,sis-04,sis-05,sis-06,sis,...,sex_male,education_label,work_part_time,fracture_femur fracture,fracture_hip fracture,fracture_hip replacement,fracture_pelvis fracture,ethnicity_black,ethnicity_caucasian,ethnicity_south asian
0,1,2022-03-31,2022-04-13,5,5,4,4,5,2,25,...,0,1,0,0,1,0,0,1,0,0
1,1,2022-04-01,2022-04-13,5,5,4,4,5,2,25,...,0,1,0,0,1,0,0,1,0,0
2,1,2022-04-02,2022-04-13,5,5,4,4,5,2,25,...,0,1,0,0,1,0,0,1,0,0
3,1,2022-04-03,2022-04-13,5,5,4,4,5,2,25,...,0,1,0,0,1,0,0,1,0,0
4,1,2022-04-04,2022-04-13,5,5,4,4,5,2,25,...,0,1,0,0,1,0,0,1,0,0


In [16]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1008 entries, 0 to 1007
Data columns (total 95 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   participant                            1008 non-null   int64  
 1   timestamp                              1008 non-null   object 
 2   clinical-timestamp                     1008 non-null   object 
 3   sis-01                                 1008 non-null   int64  
 4   sis-02                                 1008 non-null   int64  
 5   sis-03                                 1008 non-null   int64  
 6   sis-04                                 1008 non-null   int64  
 7   sis-05                                 1008 non-null   int64  
 8   sis-06                                 1008 non-null   int64  
 9   sis                                    1008 non-null   int64  
 10  ohs-01                                 1008 non-null   int64  
 11  ohs-

In [17]:
# Compute quartiles for discretization
siss_q1, siss_q2, siss_q3 = np.percentile(data["sis"], [25, 50, 75])
ohss_q1, ohss_q2, ohss_q3 = np.percentile(data["ohs"], [25, 50, 75])
okss_q1, okss_q2, okss_q3 = np.percentile(data["oks"], [25, 50, 75])

In [18]:
# Define quartile-based bins and labels
quartile_labels = [0, 1, 2, 3]

In [21]:
# Apply discretization
data["SISS_Category_Q"] = pd.cut(data["sis"], bins=[data["sis"].min(), siss_q1, siss_q2, siss_q3, data["sis"].max()],
                               labels=quartile_labels, include_lowest=True).astype(int)
data["OHSS_Category_Q"] = pd.cut(data["ohs"], bins=[data["ohs"].min(), ohss_q1, ohss_q2, ohss_q3, data["ohs"].max()],
                               labels=quartile_labels, include_lowest=True).astype(int)

data["OKSS_Category_Q"] = pd.cut(data["oks"], bins=[data["oks"].min(), okss_q1, okss_q2, okss_q3, data["oks"].max()],
                               labels=quartile_labels, include_lowest=True).astype(int)

In [22]:
# Extract only numeric features for LOPO (drop timestamps/string columns).
exclude_cols = [
    "participant",
    "timestamp",
    "clinical-timestamp",
    "motion-max-timestamp",
    "step-max-timestamp",
    "SISS_Category_Q",
    "OHSS_Category_Q",
    "OKSS_Category_Q",
]

feature_cols = [c for c in data.columns if c not in exclude_cols]
X = data[feature_cols].select_dtypes(include=[np.number]).copy()
groups = data["participant"]

# Conta i record per ogni partecipante
counts = df.groupby("participant").size()

WINDOW_SIZE = 56

In [32]:
nct = df.groupby('participant')['clinical-timestamp'].nunique()
print()
print('Visite distinte (clinical-timestamp) per partecipante:', nct.to_string())


Visite distinte (clinical-timestamp) per partecipante: participant
1     4
2     4
3     4
4     4
5     4
6     4
7     4
8     4
9     4
10    4
11    4
12    4
13    4
14    4
15    4
16    4
17    4
18    4


In [23]:
counts

participant
1     56
2     56
3     56
4     56
5     56
6     56
7     56
8     56
9     56
10    56
11    56
12    56
13    56
14    56
15    56
16    56
17    56
18    56
dtype: int64

In [43]:
# Define classifier and hyperparameter grid
param_grid = {
            "model__n_units": [16, 32, 64],
            "model__dropout": [0.0, 0.2],
            "model__lr": [1e-3, 3e-4],
            "batch_size": [16, 32],
            "epochs": [10, 20],
        }

In [40]:
# Leave-One-Patient-Out CV (LOPO)
outer_logo = LeaveOneGroupOut()

In [25]:
# Initialize results storage
overall_conf_matrix = np.zeros((4, 4))  # Assuming 4 categories (0, 1, 2, 3)
performance_metrics = []

In [33]:
# ESEMPIO D'USO:
WINDOW_SIZE = 7 # 4-14-28-56

lista_features = X.columns

X_input, y_input, groups_input = create_sequences_by_participant(
    data, WINDOW_SIZE, lista_features, ['sis', 'ohs', 'oks']
)

# Ora le lunghezze saranno TUTTE uguali e coerenti
print(X_input.shape[0], y_input.shape[0], groups_input.shape[0])

882 882 882


In [34]:
data.shape

(1008, 98)

In [35]:
X_input.shape 

(882, 7, 90)

In [36]:
len(X_input[0][0])

90

In [37]:
y_input  ### 18 pazienti e 3 var risposta

array([[25, 25, 31],
       [25, 25, 31],
       [25, 25, 31],
       ...,
       [30, 23, 32],
       [30, 23, 32],
       [30, 23, 32]], shape=(882, 3))

In [46]:
# Outer LOPO Loop
count=0
y_input = y_input 

for train_idx, test_idx in outer_logo.split(X_input, y_input, groups_input):
    #print(train_idx) index
    count=count+1
    print(count)
    
    #if count == 6 or count == 7:
    #    continue 
    X_train_outer, X_test = X_input[train_idx], X_input[test_idx] #X_train_outer, X_test = X_input.loc[train_idx].to_numpy(), X.iloc[test_idx].to_numpy()
    y_train_outer, y_test = y_input[train_idx], y_input[test_idx] #y_train_outer, y_test = y_input.iloc[train_idx].to_numpy(), y.iloc[test_idx].to_numpy()
    groups_train_outer = groups_input[train_idx] #groups_train_outer = groups.iloc[train_idx]
    print(np.unique(groups_train_outer)) #print(groups_train_outer.unique())
    
    # Inner LOPO for Hyperparameter Optimization
    inner_logo = LeaveOneGroupOut()
    best_model = None
    best_score = -np.inf
    
    for inner_train_idx, inner_val_idx in inner_logo.split(X_train_outer, y_train_outer, groups_train_outer):
        X_train_inner, X_val = X_train_outer[inner_train_idx], X_train_outer[inner_val_idx]
        y_train_inner, y_val = y_train_outer[inner_train_idx], y_train_outer[inner_val_idx]
        groups_train_inner = groups_train_outer[inner_train_idx] #groups_train_inner = groups_train_outer.iloc[inner_train_idx]
        print(np.unique(groups_train_inner)) #print(groups_train_inner.unique())
        
        # Hyperparameter tuning
        for params in ParameterGrid(param_grid):
            params = {k: int(v) if isinstance(v, np.generic) else v for k, v in params.items()}
            model = KerasClassifier(
                                    model=build_lstm_model,
                                    n_features=len(lista_features),
                                    epochs=10,
                                    batch_size=32,
                                    verbose=0
                                )
            model.set_params(**params)
            model.fit(X_train_inner, y_train_inner)
            y_val_pred = model.predict(X_val)
            score = f1_score(y_val, y_val_pred, average="macro")
            # https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html

            if score > best_score:
                best_score = score
                best_model = model
                best_params = params  # Store the best hyperparameters

    # Train best model on full outer training set
    best_model.fit(X_train_outer, y_train_outer)
    y_pred = best_model.predict(X_test)

    # Compute metrics
    f1 = f1_score(y_test, y_pred, average="macro")
    balanced_acc = balanced_accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred, average="macro")
    precision = precision_score(y_test, y_pred, average="macro")
    conf_matrix = confusion_matrix(y_test, y_pred, labels=[0, 1, 2, 3])

    # Aggregate confusion matrices
    overall_conf_matrix += conf_matrix

    # Store results
    performance_metrics.append([f1, balanced_acc, recall, precision])

1
[ 2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18]
[ 3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18]


KeyboardInterrupt: 

In [ ]:
overall_conf_matrix

In [ ]:
# Convert results to DataFrame
performance_df = pd.DataFrame(performance_metrics, columns=["Macro-F1", "Balanced Accuracy", "Macro Recall", "Macro Precision"])

In [ ]:
performance_df

In [ ]:
assert False

In [ ]:
# Save performance metrics and overall confusion matrix
output_path = os.path.join(root, "results/model_performance_ohss.xlsx")
with pd.ExcelWriter(output_path) as writer:
    performance_df.to_excel(writer, sheet_name="All_Folds")

In [ ]:
# Plot overall confusion matrix
plt.figure(figsize=(6, 5))
sns.heatmap(overall_conf_matrix, annot=True, cmap="coolwarm", xticklabels=[0, 1, 2, 3], yticklabels=[0, 1, 2, 3])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Overall Confusion Matrix OHSS")
conf_matrix_path = os.path.join(root, "results/overall_confusion_matrix_ohss.pdf")
plt.savefig(conf_matrix_path, dpi=300, bbox_inches='tight')
plt.close()

print(f"\n✅ Performance metrics saved as: {output_path}")
print(f"✅ Overall Confusion Matrix saved as: {conf_matrix_path}")